<a href="https://colab.research.google.com/github/notmakgg-ux/GPT-build-by-me/blob/main/TrainingMyGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Building my own GPT from scratch.
 * Today I am learning and building my own GPT from scratch. This will help me
learn the base needed for LLM.
* The flow of the GPT is as follows :-

1.   Tokenization
2.   Embedding
3. Positional encoding
4. Attention
5. Transformer Block
6. GPT model
7. Training
8. Evaluation



## 1. Tokenization

In [ ]:
from dataclasses import dataclass
import tiktoken

In [ ]:
@dataclass
class TokenizerConfig:
    name: str = "gpt2"
    vocab_size: int = 50257

In [ ]:
class SimpleTokenizer:
  def __init__(self, config=None):
    self.config=config or TokenizerConfig()
    self.enc= tiktoken.get_encoding(self.config.name)
    self.eos_token="<|endoftext|>"
    self.eos_token_id=self.enc.encode(
        self.eos_token,  allowed_special={self.eos_token}
    )[0]
  def encode(self, text):
    return self.enc.encode(text , allowed_special={self.eos_token})

  def decode(self,ids):
    return self.enc.decode(ids)

  @property
  def vocab_size(self):
    return self.config.vocab_size

<>:5: SyntaxWarning: invalid escape sequence '\>'
<>:5: SyntaxWarning: invalid escape sequence '\>'
/tmp/ipykernel_6988/3421483214.py:5: SyntaxWarning: invalid escape sequence '\>'
  self.eos_token="<|endoftext\>"


In [ ]:
tokenizer = SimpleTokenizer()
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"EOS token: {tokenizer.eos_token}")
print(f"EOS token ID: {tokenizer.eos_token_id}")
print()

text="I am a Gangstar with a gun."
print(f"Original text: {text}")
encoded = tokenizer.encode(text)
print(f"Encoded text: {encoded}")
decoded = tokenizer.decode(encoded)
print(f"Decoded text: {decoded}")
print(f"Roundtrip: {text == decoded}")


Vocab size: 50257
EOS token: <|endoftext\>
EOS token ID: 27

Original text: I am a Gangstar with a gun.
Encoded text: [40, 716, 257, 19228, 7364, 351, 257, 2485, 13]
Decoded text: I am a Gangstar with a gun.
Roundtrip: True


In [ ]:
rare = tokenizer.encode("antidisestablishmentarianism")
pieces = [tokenizer.decode([t]) for t in rare]
print(f"Pieces: {pieces}")
print(f"Decoded: '{tokenizer.decode(rare)}'")
print()

text = tokenizer.encode("Hello world!")
print(f"Decoded: {tokenizer.decode(text)}")
print()
emoji = tokenizer.encode("Hello \U0001f60a world")
print(f"Emoji test: {tokenizer.decode(emoji)}")

Pieces: ['ant', 'idis', 'establishment', 'arian', 'ism']
Decoded: 'antidisestablishmentarianism'

Decoded: Hello world!

Emoji test: Hello 😊 world


## 2. Embedding

In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
class Embedding(nn.Module):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.embed=nn.Embedding(vocab_size,d_model)

  def forward(self,x):
    return self.embed(x)


In [ ]:
vocab_size = 1000
d_model = 768

embedding = Embedding(vocab_size, d_model)
print(f"Embedding table shape: [{vocab_size}, {d_model}]")
print(f"Total parameters: {sum(p.numel() for p in embedding.parameters()):,}")
print()

token_ids = torch.tensor([[123, 45, 678, 464, 890]])
print(f"Input token IDs: {token_ids}")
print(f"Input shape: {token_ids.shape}")
print()

output = embedding(token_ids)
print(f"Output shape: {output.shape}")
print(f"Each token is now a {d_model}-dimensional vector")
print(f"First token vector preview: {output[0, 0, :5].tolist()} ...")

Embedding table shape: [1000, 768]
Total parameters: 768,000

Input token IDs: tensor([[123,  45, 678, 464, 890]])
Input shape: torch.Size([1, 5])

Output shape: torch.Size([1, 5, 768])
Each token is now a 768-dimensional vector
First token vector preview: [0.389218270778656, -0.25960472226142883, 1.7515723705291748, -1.1836599111557007, -0.49302583932876587] ...


## 3. Postional Encoding Using RoPE

In [1]:
import torch
import torch.nn as nn
import math

In [12]:
class RotaryPositionalEmbedding(nn.Module):
  def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
    super().__init__()
    assert d_model%2==0, "d_model must be even"


    dim_indices=torch.arange(0, d_model, 2).float()
    inv_freq=1.0 / (theta ** (dim_indices / d_model))

    poitions = torch.arange(max_seq_len).float()
    freqs=torch.outer(poitions, inv_freq)
    emb = freqs.repeat_interleave(2, dim=-1)

    self.register_buffer("cos_cached", emb.cos())
    self.register_buffer("sin_cached", emb.sin())


  @staticmethod
  def rotate_half(x):
      x_even = x[..., 0::2]
      x_odd  = x[..., 1::2]
      return torch.stack([-x_odd, x_even], dim=-1).flatten(-2)
  def forward(self, x, seq_len):
      cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
      sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
      return (x * cos) + (self.rotate_half(x) * sin)





In [13]:
rope = RotaryPositionalEmbedding(d_model=4, max_seq_len=16)

q = torch.tensor([[[[0.8, 0.3, -0.5, 0.2]]]])  # [1, 1, 1, 4]
print(f"Original vector: {q[0, 0, 0].tolist()}")
print()

for pos in [0, 1, 2, 5]:
    rotated = rope(q, seq_len=pos + 1)
    last = rotated[0, 0, -1].tolist()
    vals = [f"{v:.3f}" for v in last]
    print(f"Position {pos}: [{', '.join(vals)}]")

Original vector: [0.800000011920929, 0.30000001192092896, -0.5, 0.20000000298023224]

Position 0: [0.800, 0.300, -0.500, 0.200]
Position 1: [0.180, 0.835, -0.502, 0.195]
Position 2: [-0.606, 0.603, -0.504, 0.190]
Position 5: [0.515, -0.682, -0.509, 0.175]


In [14]:

rope = RotaryPositionalEmbedding(d_model=64, max_seq_len=256)

seq_len = 12
num_heads = 4
head_dim = 64

q = torch.randn(1, num_heads, seq_len, head_dim)
k = torch.randn(1, num_heads, seq_len, head_dim)

q_rot = rope(q, seq_len=seq_len)
k_rot = rope(k, seq_len=seq_len)

head = 0
a = (q_rot[0, head, 2] @ k_rot[0, head, 4]).item()
b = (q_rot[0, head, 5] @ k_rot[0, head, 7]).item()
c = (q_rot[0, head, 0] @ k_rot[0, head, 8]).item()

print(f"Distance 2 (positions 2,4):     {a:.4f}")
print(f"Distance 2 (positions 5,7):     {b:.4f}")
print(f"Distance 8 (positions 0,8):     {c:.4f}")
print()

Distance 2 (positions 2,4):     10.2846
Distance 2 (positions 5,7):     0.6078
Distance 8 (positions 0,8):     -23.4536



## 4. Attention

In [15]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [16]:
def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)

In [31]:

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()

        assert d_model % num_heads == 0, \
            "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.rope = RotaryPositionalEmbedding(self.head_dim)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.shape


        qkv = self.qkv_proj(x)

       s
        qkv = qkv.reshape(
            batch_size, seq_len, 3, self.num_heads, self.head_dim
        )
        qkv = qkv.permute(2, 0, 3, 1, 4)

        q, k, v = qkv[0], qkv[1], qkv[2]


        q = self.rope(q, seq_len)
        k = self.rope(k, seq_len)


        attn_weights = (
            q @ k.transpose(-2, -1)
        ) / math.sqrt(self.head_dim)


        if mask is not None:
            attn_weights = attn_weights.masked_fill(
                mask == 0, float("-inf")
            )


        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)


        attn_output = attn_weights @ v


        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(
            batch_size, seq_len, self.d_model
        )


        output = self.out_proj(attn_output)
        output = self.resid_dropout(output)

        return output

In [32]:
attn = MultiHeadAttention(d_model=768, num_heads=12)

batch_size = 8
seq_len = 64
x = torch.randn(batch_size, seq_len, 768)
mask = create_causal_mask(seq_len, x.device)

output = attn(x, mask)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Input and output shapes match: {x.shape == output.shape}")
print()

params = sum(p.numel() for p in attn.parameters())
print(f"Attention parameters: {params:,}")


Input shape:  torch.Size([8, 64, 768])
Output shape: torch.Size([8, 64, 768])
Input and output shapes match: True

Attention parameters: 2,359,296


In [33]:
tiny_attn = MultiHeadAttention(d_model=4, num_heads=1)

tokens = torch.tensor([[
    [0.5,  0.2, -0.3,  0.8],
    [0.1, -0.5,  0.7, -0.2],
    [0.9,  0.3, -0.1, -0.5],
]])
print(f"Input tokens shape: {tokens.shape}")
print()

with torch.no_grad():
    qkv = tiny_attn.qkv_proj(tokens)
    qkv = qkv.reshape(1, 3, 3, 1, 4)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]

print("Query vectors:")
for i in range(3):
    vals = [f"{v:.3f}" for v in q[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")
print()

print("Key vectors:")
for i in range(3):
    vals = [f"{v:.3f}" for v in k[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")
print()

scores = (q @ k.transpose(-2, -1)) / math.sqrt(4)
print("Attention scores (before mask and softmax):")
print(f"  Token 0: {[f'{v:.3f}' for v in scores[0, 0, 0].tolist()]}")
print(f"  Token 1: {[f'{v:.3f}' for v in scores[0, 0, 1].tolist()]}")
print(f"  Token 2: {[f'{v:.3f}' for v in scores[0, 0, 2].tolist()]}")

Input tokens shape: torch.Size([1, 3, 4])

Query vectors:
  Token 0: [0.061, 0.086, -0.383, -0.329]
  Token 1: [-0.015, -0.096, 0.188, 0.366]
  Token 2: [-0.131, 0.422, 0.394, -0.104]

Key vectors:
  Token 0: [-0.398, -0.063, -0.337, 0.128]
  Token 1: [0.009, 0.302, 0.286, 0.134]
  Token 2: [0.241, 0.335, -0.079, 0.265]

Attention scores (before mask and softmax):
  Token 0: ['0.029', '-0.064', '-0.007']
  Token 1: ['-0.002', '0.037', '0.023']
  Token 2: ['-0.060', '0.113', '0.026']


In [34]:
mask = create_causal_mask(3, scores.device)
scores_masked = scores.masked_fill(mask == 0, float('-inf'))
weights = F.softmax(scores_masked, dim=-1)

print("Attention weights (after mask and softmax):")
for i in range(3):
    row = [f"{v:.3f}" for v in weights[0, 0, i].tolist()]
    print(f"  Token {i} attends to: [{', '.join(row)}]")
print()
print("Token 0 sees only itself. Token 1 sees 0 and 1. Token 2 sees all three.")
print()

output = weights @ v
print("Output vectors (weighted sum of values):")
for i in range(3):
    vals = [f"{v:.3f}" for v in output[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")

Attention weights (after mask and softmax):
  Token 0 attends to: [1.000, 0.000, 0.000]
  Token 1 attends to: [0.490, 0.510, 0.000]
  Token 2 attends to: [0.305, 0.363, 0.332]

Token 0 sees only itself. Token 1 sees 0 and 1. Token 2 sees all three.

Output vectors (weighted sum of values):
  Token 0: [0.044, -0.131, 0.158, -0.240]
  Token 1: [0.058, 0.090, 0.048, 0.063]
  Token 2: [0.193, 0.177, 0.228, -0.074]


In [35]:
print("Causal attention matrix (Token i attending to Token j):")
print("          T0      T1      T2")
for i in range(3):
    row = "  ".join([f"{weights[0, 0, i, j]:.3f}" for j in range(3)])
    print(f"T{i}       {row}")
print()
print("The upper right is zero. The model cannot see the future.")

Causal attention matrix (Token i attending to Token j):
          T0      T1      T2
T0       1.000  0.000  0.000
T1       0.490  0.510  0.000
T2       0.305  0.363  0.332

The upper right is zero. The model cannot see the future.


## 5. Transformer             

In [36]:
class RMSNorm(nn.Module):
  def __init__(self,d_model,eps=1e-6):
    super().__init__()
    self.weight=nn.Parameter(torch.ones(d_model))
    self.eps=eps

  def forward(self,x):
    rms= torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
    return x * self.weight * rms

In [38]:
class SwiGLU(nn.Module):
  def __init__(self,d_model, expansion_factor=4):
    super().__init__()
    hidden_dim = expansion_factor * d_model
    self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
    self.w2 = nn.Linear(d_model, hidden_dim, bias=False)
    self.w3 = nn.Linear(hidden_dim, d_model, bias=False)
  def forward(self, x):
    return self.w3(F.silu(self.w1(x)) * self.w2(x))


In [39]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, mask=None):
        x = x + self.attention(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

In [40]:
d_model = 768
num_heads = 12
seq_len = 64
batch_size = 4

block = TransformerBlock(d_model, num_heads)

x = torch.randn(batch_size, seq_len, d_model)
mask = create_causal_mask(seq_len, x.device)

output = block(x, mask)

print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Shapes match: {x.shape == output.shape}")
print()

diff = (output - x).abs().mean().item()
print(f"Mean change after block: {diff:.4f}")
print(f"The block changed the input (it did something)")
print()

params = sum(p.numel() for p in block.parameters())
print(f"Parameters per block: {params:,}")
print(f"For a 12 layer GPT-2 Small: {params * 12:,} params in all blocks")

Input shape:  torch.Size([4, 64, 768])
Output shape: torch.Size([4, 64, 768])
Shapes match: True

Mean change after block: 0.1121
The block changed the input (it did something)

Parameters per block: 9,438,720
For a 12 layer GPT-2 Small: 113,264,640 params in all blocks


In [41]:
block.eval()
with torch.no_grad():
    for p in block.attention.parameters():
        p.zero_()
    for p in block.ffn.parameters():
        p.zero_()

x = torch.randn(1, 4, d_model)
mask = create_causal_mask(4, x.device)
output = block(x, mask)

diff = (output - x).abs().max().item()
print(f"Max difference when weights are zero: {diff:.10f}")
print(f"Input passes through unchanged (the residual works)")

Max difference when weights are zero: 0.0000000000
Input passes through unchanged (the residual works)
